# GES Employment Outcomes — Data Cleaning
**Assigned to:** Tahmin
**Dataset:** `ges_employment_outcomes.csv` (from CA2 Datasets, Brightspace)
**Purpose:** Clean this dataset so it can be merged with Ben's `ges_degree_details.csv` on the shared `Key` column. Together they form the "combined Brightspace dataset" required by the assignment brief (Section 2, Requirement 4).

This notebook only cleans `ges_employment_outcomes.csv`. The merge with `ges_degree_details` is previewed at the end, but the final merged dataset used for analysis should be built once in the group's main analysis notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)

## 1. Load the dataset

Loading the raw file exactly as provided, with no assumptions about dtypes yet, so we can see the real state of the data first.

In [ ]:
raw_path = "1784698367215_ges_employment_outcomes.csv"  # rename to match your local copy if needed
df = pd.read_csv(raw_path)

print("Shape:", df.shape)
df.head()

## 2. Initial inspection

Checking structure, dtypes, and nulls before doing anything else.

In [ ]:
df.info()

In [ ]:
# How many columns are entirely empty?
empty_cols = df.columns[df.isna().all()]
print(f"Fully empty columns: {len(empty_cols)}")
print(list(empty_cols))

**Finding:** The raw CSV has 48 columns, but only 12 contain any data. The other columns (mostly `Unnamed: 12` through `Unnamed: 46`) are artifacts of trailing commas in the source file and are completely empty. These will be dropped in Step 4.

## 3. Remove embedded duplicate header row(s)

This dataset appears to have been assembled by stacking multiple yearly tables. Occasionally, a header row from one of the original tables got pulled in as a data row. We detect this by checking whether a row's value equals its own column name.

In [ ]:
# A genuine data row should never contain the literal column name as its value.
# Flag any row where the first data column's value equals its own column header.
header_mask = df["employment_rate_overall"] == "employment_rate_overall"
print(f"Embedded header rows found: {header_mask.sum()}")
df[header_mask]

In [ ]:
df = df[~header_mask].reset_index(drop=True)
print("Shape after removing embedded header row(s):", df.shape)

## 4. Drop empty and duplicate columns

- The `Unnamed: *` columns are completely empty (100% null) — dropped.
- `gross_monthly_mean.1`, `gross_monthly_median.1`, `gross_mthly_25_percentile.1`, and `gross_mthly_75_percentile.1` are exact duplicates of their non-`.1` counterparts (verified below) — dropped.

In [ ]:
# Confirm the .1 columns are true duplicates before dropping them
dup_pairs = [
    ("gross_monthly_mean", "gross_monthly_mean.1"),
    ("gross_monthly_median", "gross_monthly_median.1"),
    ("gross_mthly_25_percentile", "gross_mthly_25_percentile.1"),
    ("gross_mthly_75_percentile", "gross_mthly_75_percentile.1"),
]

for a, b in dup_pairs:
    match_rate = (df[a].fillna("MISSING") == df[b].fillna("MISSING")).mean()
    print(f"{a} vs {b}: {match_rate:.2%} identical")

In [ ]:
# Drop fully empty columns + confirmed duplicate columns
cols_to_drop = list(empty_cols) + [b for _, b in dup_pairs]
df = df.drop(columns=cols_to_drop)

print("Shape after dropping empty/duplicate columns:", df.shape)
df.head()

## 5. Standardise missing value markers

Missing values in this dataset are stored as the literal string `'na'` rather than a true null. This is why every numeric column was read in as text (`object`) dtype instead of numeric. We replace `'na'` with proper `NaN`.

In [ ]:
outcome_cols = [
    "employment_rate_overall",
    "employment_rate_ft_perm",
    "basic_monthly_mean",
    "basic_monthly_median",
    "gross_monthly_mean",
    "gross_monthly_median",
    "gross_mthly_25_percentile",
    "gross_mthly_75_percentile",
]

df[outcome_cols] = df[outcome_cols].replace("na", np.nan)

## 6. Convert columns to correct numeric dtype

Now that `'na'` has been replaced with real `NaN`, these columns can be safely converted to numeric types.

In [ ]:
for col in outcome_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[outcome_cols].dtypes

In [ ]:
df.info()

## 7. Check for duplicate rows

`Key` should be unique (it's the join key to `ges_degree_details`), so we check both full-row duplicates and duplicate Keys.

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate Key values:", df['Key'].duplicated().sum())

## 8. Handle missing values

Each of the 8 outcome columns has the same number of missing values. This is expected: SkillsFuture/GES suppresses salary and employment rate figures for degree cohorts with very small graduating numbers (to protect individual privacy), rather than these being data entry errors.

**Decision:** We keep these rows with `NaN` rather than dropping them, for two reasons:
1. Dropping rows here would break the row-alignment with `ges_degree_details.csv` on `Key`, since that dataset has no missing rows to match against.
2. The missingness itself is informative (small/niche cohorts) — imputing a salary figure would misrepresent those cohorts. Rows with missing values can be filtered out later, at analysis time, only for the specific columns/visualisations that need complete data (e.g. `.dropna(subset=[...])` before a boxplot or KDE plot).

In [ ]:
missing_summary = df[outcome_cols].isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary

## 9. Outlier check (boxplot)

A quick boxplot of the salary columns to see the spread and flag any extreme outliers before this data gets used downstream. We don't remove outliers here — legitimately high or low salaries are real and relevant to the research question — but it's worth visualising and noting.

In [ ]:
salary_cols = ["basic_monthly_mean", "basic_monthly_median", "gross_monthly_mean", "gross_monthly_median"]

plt.figure(figsize=(10, 6))
sns.boxplot(data=df[salary_cols].dropna())
plt.title("Distribution of Monthly Salary Figures (Cleaned Data)")
plt.ylabel("S$ per month")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Final summary statistics

In [ ]:
df[outcome_cols].describe()

## 11. Save cleaned dataset

Saving the cleaned dataset so it can be merged with `ges_degree_details.csv` (Ben's cleaned output) later, using `Key` as the join column.

In [ ]:
df.to_csv("ges_employment_outcomes_cleaned.csv", index=False)
print("Saved: ges_employment_outcomes_cleaned.csv")
print("Final shape:", df.shape)

## 12. Preview merge with `ges_degree_details` (Ben's dataset)

This is just a preview to confirm the two cleaned datasets join correctly on `Key`. The group's main analysis notebook should perform the real merge once both cleaned CSVs are finalised.

In [ ]:
degree_details_path = "1784698367214_ges_degree_details.csv"  # Ben's cleaned file, once ready
degree_details = pd.read_csv(degree_details_path)

merged_preview = degree_details.merge(df, on="Key", how="inner")
print("Merged shape:", merged_preview.shape)
merged_preview.head()

## Summary of cleaning steps performed

1. Loaded the raw CSV.
2. Detected and removed 1 embedded duplicate header row.
3. Dropped 35 fully empty `Unnamed` columns (artifacts of trailing commas in the source file).
4. Dropped 4 duplicate columns (`.1` suffix versions, verified identical to their originals).
5. Replaced the `'na'` string placeholder with proper `NaN` across all 8 outcome columns.
6. Converted all 8 outcome columns from text to numeric dtype.
7. Verified no duplicate rows and no duplicate `Key` values.
8. Documented and retained missing values (small-cohort data suppression) rather than dropping or imputing them.
9. Visualised salary distributions with a boxplot to check for outliers.
10. Saved the cleaned dataset as `ges_employment_outcomes_cleaned.csv`.
11. Verified the cleaned dataset merges correctly with `ges_degree_details` on `Key`.